# Simulation 3 — Binomial GLLVM, size sweep

The **symmetric twin of [simulation 1](../simulation_1/experiment.ipynb)**: same
size sweep, but responses are **Binomial** (logit link, `N=10` trials) instead of
Poisson — a bounded-count twin of the unbounded-count Poisson sweep.

| | |
|---|---|
| latent dim | $q = 2$ |
| responses | $p \in \{10, 20, 50, 100\}$ |
| observations | $n \in \{20, 100, 500\}$ |
| | $\Rightarrow$ **12 settings**, each repeated $H$ times |

**Why Binomial(10), not Bernoulli.** A single-trial GLLVM is *not identifiable* at
small $n$: loadings can drive logits to $\pm\infty$ and perfectly **separate** the
0/1s, so the likelihood is unbounded and the unregularised ZQE estimate runs away
(gllvm survives only via its variational prior). With $N>1$ trials and intermediate
probabilities the counts can't be separated → bounded likelihood → identifiable.
Verified at p=50,n=20: N=1 → 2.39 (diverges), N=3 → 0.73, **N=10 → 0.32**. This is
an identifiability fix in the *design*, not a regulariser.

Changes vs sim 1: `BinomialGLM(total_count=10)`; `T(y)=4(y/N−0.5)`; Gaussian
encoder on that `T`. **Methods**: `zqe` vs `gllvm` (`family="binomial", Ntrials=10`).

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
if not os.path.exists(os.path.join(HERE, "sweep.py")):
    HERE = os.path.join(os.getcwd(), "simulations", "simulation_3")
sys.path.insert(0, HERE)
import torch, sweep
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device  :", DEV)
print("q       :", sweep.Q, "| p grid:", sweep.P_GRID, "| n grid:", sweep.N_GRID)
print("N trials:", sweep.BINOM_TRIALS, "| wz_scale:", sweep.WZ_SCALE,
      "| family: Binomial(logit), T=4(y/N-0.5)")

## Data check: probabilities mixed, counts not saturated?

In [ ]:
import numpy as np, matplotlib.pyplot as plt
torch.manual_seed(0)
from gllvm.simulations import make_mixed, simulate
gt = make_mixed(n_latent=sweep.Q, binomial=50, binom_trials=sweep.BINOM_TRIALS, wz_scale=sweep.WZ_SCALE)
Y, _ = simulate(gt, n_samples=2000, device="cpu")
with torch.no_grad():
    base = torch.sigmoid(gt.bias).numpy()
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.2))
a.hist(base, bins=15); a.set_title("baseline P(success) per response"); a.set_xlabel("prob")
b.hist(Y.numpy().ravel(), bins=np.arange(sweep.BINOM_TRIALS + 2) - 0.5)
b.set_title(f"counts (0..{sweep.BINOM_TRIALS})"); b.set_xlabel("y")
print("baseline prob quantiles:", np.round(np.quantile(base, [.05,.25,.5,.75,.95]), 2))
print("count range:", int(Y.min()), "-", int(Y.max()), "| mean:", round(float(Y.float().mean()), 2))
plt.tight_layout(); plt.show()

## Demo: one rep (p=10, n=100, not saved)

In [ ]:
from gllvm.r_gllvm import RGllvm
r = RGllvm(method="VA", family="binomial", ntrials=sweep.BINOM_TRIALS, link="logit")
assert r.available(), f"R not found at {r.rscript!r}"
demo = sweep.run_setting(q=2, p=10, n=100, seed=0, device=DEV, rgllvm=r)
demo.drop_duplicates("method")[["method", "failed", "time_sec", "converged", "procrustes"]]

## Run the sweep (resumable)

In [ ]:
sweep.run_sweep(reps=20, device=DEV)

## Overview

In [ ]:
import pandas as pd
df = sweep.load_results()
fits = (df[df.method != "true"].drop_duplicates(["p", "n", "rep", "method"])
        [["p", "n", "rep", "method", "failed", "time_sec", "procrustes"]])
print("failures:", int(fits.failed.sum()), "/", len(fits))
(fits.groupby(["p", "n", "method"])
 .agg(procrustes=("procrustes", "mean"), time_s=("time_sec", "mean"),
      fails=("failed", "sum")).round(3))